# FAA Wildlife Strikes – Hypothesis Testing
This notebook supplements the exploratory analysis with formal statistical inference. Each test includes its hypotheses, assumptions, and reproducible code.

In [4]:
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import glm, logit
from statsmodels.stats.proportion import proportions_ztest
pd.options.display.float_format = "{:.4f}".format
DATA_DIR = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"
csv_path = DATA_DIR / "wildlife-strikes" / "database.csv"
if not csv_path.exists():
    raise FileNotFoundError("Run notebooks/download_wildlife.ipynb to populate data/wildlife-strikes/database.csv")
important_cols = [
    "Incident Year","Flight Phase","Aircraft Damage","State","Species Name",
    "Height","Speed","Distance","Engines","Species Quantity"
]
lazy = pl.scan_csv(
    csv_path,
    infer_schema_length=10000,
    null_values=["","NA"],
)
lazy = lazy.select([pl.col(c) for c in important_cols])
df = pd.DataFrame(lazy.collect().to_dicts())
df["Incident Year"] = pd.to_numeric(df["Incident Year"], errors="coerce")
df["Height"] = pd.to_numeric(df["Height"], errors="coerce")
df["Speed"] = pd.to_numeric(df["Speed"], errors="coerce")
df["Distance"] = pd.to_numeric(df["Distance"], errors="coerce")
df["Engines"] = pd.to_numeric(df["Engines"], errors="coerce")
df["Species Quantity"] = pd.to_numeric(df["Species Quantity"], errors="coerce")
df["damage_flag"] = pd.to_numeric(df["Aircraft Damage"], errors="coerce").fillna(0).clip(0,1).astype(int)
df.dropna(subset=["Incident Year"], inplace=True)
df.head()

,Incident Year,Flight Phase,Aircraft Damage,State,Species Name,Height,Speed,Distance,Engines,Species Quantity,damage_flag
0,1990,CLIMB,1,KY,GULL,NaN,NaN,NaN,2.0000,1.0000,1
1,1990,TAKEOFF RUN,0,HI,HOUSE SPARROW,0.0000,NaN,0.0000,2.0000,1.0000,0
2,1990,None,0,HI,BARN OWL,NaN,NaN,0.0000,NaN,1.0000,0
3,1990,APPROACH,0,SC,UNKNOWN MEDIUM BIRD,200.0000,138.0000,NaN,2.0000,1.0000,0
4,1990,CLIMB,0,FL,FINCH,100.0000,200.0000,NaN,NaN,1.0000,0


## 1. Temporal Trend in Reported Strikes
**Goal:** test whether annual strike counts follow an increasing trend.
**Hypotheses**
- H₀: The log-rate of strikes does **not** depend on `Incident Year` (slope = 0).
- H₁: The log-rate of strikes **increases** with `Incident Year` (slope > 0).
**Assumptions**
1. Yearly counts are Poisson-distributed conditional on the mean.
2. Counts from different years are independent.
3. Exposure (overall flight opportunity) is roughly constant or its changes are absorbed by the year effect.

In [5]:
year_counts = (
    df.groupby("Incident Year").size().reset_index(name="strike_count")
)
year_counts["centered_year"] = year_counts["Incident Year"] - year_counts["Incident Year"].mean()
poisson_model = glm(
    formula="strike_count ~ centered_year",
    data=year_counts,
    family=sm.families.Poisson(),
)
poisson_result = poisson_model.fit()
poisson_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:           strike_count   No. Observations:                   26
Model:                            GLM   Df Residuals:                       24
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1378.6
Date:                Wed, 03 Dec 2025   Deviance:                       2483.8
Time:                        17:35:29   Pearson chi2:                 2.43e+03
No. Iterations:                     4   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         8.6901      0.003   3235.473      0.000       8.685       8.695
centered_year     0.0659      0.000    192.159      0.000       0.065       0.067
=================================================================================
"""

A one-sided p-value for the slope can be obtained from the GLM summary (`centered_year`). Reject H₀ if the slope coefficient is significantly positive.

## 2. State vs. Damage Severity (Chi-square Test)
**Hypotheses**
- H₀: `State` and damage occurrence are independent.
- H₁: Damage probability varies by state.
**Assumptions**
1. Each strike is counted once (independent observations).
2. Expected counts per cell are ≥ 5 (we restrict to the 10 highest-volume states).
3. Sample represents the population of interest over the study period.

In [6]:
state_order = df["State"].value_counts().head(10).index
state_subset = df[df["State"].isin(state_order)].copy()
contingency = pd.crosstab(state_subset["State"], state_subset["damage_flag"])
chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
chi2, dof, p_value

(np.float64(488.6916088918108), 9, np.float64(1.5155773906215418e-99))

The χ² statistic, degrees of freedom, and p-value summarize whether damage rates differ significantly among the ten busiest states.

## 3. Species Group vs. Damage Probability (Logistic Regression)
**Hypotheses**
- H₀: After grouping species, log-odds of damage do not depend on species (all coefficients = 0).
- H₁: At least one species group has a different log-odds of damage than the baseline.
**Assumptions**
1. Observations are independent.
2. Logit link is appropriate (probabilities between 0 and 1; linear predictor suffices).
3. No severe multicollinearity among predictors (here: categorical species indicator only).

In [7]:
top_species = df["Species Name"].value_counts().head(5).index
species_subset = df[df["Species Name"].notna()].copy()
species_subset["species_group"] = np.where(
    species_subset["Species Name"].isin(top_species),
    species_subset["Species Name"],
    "OTHER",
)
logit_data = species_subset[["damage_flag","species_group"]].copy()
logit_result = logit("damage_flag ~ C(species_group)", data=logit_data).fit(disp=False)
logit_result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            damage_flag   No. Observations:               174024
Model:                          Logit   Df Residuals:                   174018
Method:                           MLE   Df Model:                            5
Date:                Wed, 03 Dec 2025   Pseudo R-squ.:                 0.03059
Time:                        17:35:57   Log-Likelihood:                -49462.
converged:                       True   LL-Null:                       -51023.
Covariance Type:            nonrobust   LLR p-value:                     0.000
===========================================================================================================
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                  -1.6113      0.033    -48.680      0.000      -1.676      -1.546
C(species_group)[T.MOURNING DOVE]          -1.9665      0.078    -25.308      0.000      -2.119      -1.814
C(species_group)[T.OTHER]                  -0.6084      0.035    -17.321      0.000      -0.677      -0.540
C(species_group)[T.UNKNOWN BIRD]           -0.9826      0.059    -16.585      0.000      -1.099      -0.867
C(species_group)[T.UNKNOWN MEDIUM BIRD]    -0.4783      0.037    -12.967      0.000      -0.551      -0.406
C(species_group)[T.UNKNOWN SMALL BIRD]     -1.9046      0.047    -40.818      0.000      -1.996      -1.813
===========================================================================================================
"""

Inspect the coefficient table to see which species differ significantly from the baseline ("OTHER") group.

## 4. Pairwise Comparison of Flight Phases (Proportion z-test)
**Example Hypothesis:** compare damage rates between `EN ROUTE` and `TAKEOFF RUN`.
- H₀: p₁ − p₂ = 0 (equal damage proportions).
- H₁: p₁ − p₂ > 0 (en-route strikes have higher damage probability).
**Assumptions**
1. Binomial sampling with sufficiently large sample sizes (np ≥ 5 and n(1−p) ≥ 5).
2. Samples from the two phases are independent.
3. Observations within each phase are independent.

In [8]:
phase_a, phase_b = "EN ROUTE", "TAKEOFF RUN"
phase_counts = (
    df[df["Flight Phase"].isin([phase_a, phase_b])]
    .groupby("Flight Phase")
    .agg({"damage_flag":"sum","Flight Phase":"count"})
    .rename(columns={"damage_flag":"damages","Flight Phase":"total"})
)
successes = phase_counts.loc[[phase_a, phase_b], "damages"].to_numpy()
samples = phase_counts.loc[[phase_a, phase_b], "total"].to_numpy()
z_stat, p_val = proportions_ztest(successes, samples, alternative="larger")
phase_counts, z_stat, p_val

(              damages  total
 Flight Phase                
 EN ROUTE         1155   2989
 TAKEOFF RUN      2017  21953,
 np.float64(45.34424570686256),
 np.float64(0.0))

Reject H₀ in favor of higher en-route damage probability if the one-sided p-value is below the chosen α.

## 5. Height vs. Distance Correlation (Pearson r)
**Hypotheses**
- H₀: ρ = 0 (no linear correlation between strike height and reported distance).
- H₁: ρ ≠ 0.
**Assumptions**
1. Paired (Height, Distance) observations are independent.
2. Joint distribution is approximately bivariate normal (or sample size is large enough for robustness).
3. Measurement scales are continuous and at least interval-level.

In [9]:
corr_sample = df.dropna(subset=["Height","Distance"])
pearson_r, pearson_p = stats.pearsonr(corr_sample["Height"], corr_sample["Distance"] )
pearson_r, pearson_p

(np.float64(0.856572003844047), np.float64(0.0))

If |r| is large and p < α, we conclude linear association exists between altitude and distance at strike time.

## 6. Mean Airspeed vs. Damage Outcome (Welch’s t-test)
**Hypotheses**
- H₀: Mean speed is equal for damaged and non-damaged strikes.
- H₁: Mean speed differs between the two groups.
**Assumptions**
1. Two groups are independent.
2. Within-group speed distributions are approximately normal (t-test is fairly robust for large samples).
3. Observations are measured on a continuous scale; Welch’s variant allows unequal variances.

In [10]:
speed_sample = df.dropna(subset=["Speed"])
damaged_speed = speed_sample.loc[speed_sample["damage_flag"] == 1, "Speed"]
undamaged_speed = speed_sample.loc[speed_sample["damage_flag"] == 0, "Speed"]
t_stat, t_p = stats.ttest_ind(damaged_speed, undamaged_speed, equal_var=False, nan_policy="omit")
{"t_stat": t_stat, "p_value": t_p, "damaged_mean": damaged_speed.mean(), "undamaged_mean": undamaged_speed.mean()}

{'t_stat': np.float64(0.30921032703253026),
 'p_value': np.float64(0.7571675980669681),
 'damaged_mean': np.float64(142.0889066423191),
 'undamaged_mean': np.float64(141.9027297747056)}

Each section reports the key statistic and p-value; compare them to your preferred significance threshold (e.g., α = 0.05) to make decisions, remembering to adjust for multiple comparisons if interpreting all results jointly.